# Cellpose 細胞ROI抽出 (Google Colab版)

研究室のGPU付きPCが故障したため、**Cellposeによる細胞セグメンテーション〜ROI (ImageJ形式) の書き出し**をGoogle Colab上で行うためのノートブックです。

撮影データはZスタック（1回の撮影につき `_C001Z001` 〜 `_C001Z007` のようなZスライスtifが複数枚）になっており、撮影ごとにフォルダが分かれている構成を前提としています。**Zスライスごとに個別にセグメンテーション・ROI作成を行い**、どのZが一番ピントが合っているかは後でFiji上で選んで使う運用です（このノートブックではZ投影は行いません）。

## 全体の流れ

1. (このノートブック) Google Drive上の親フォルダ以下を再帰的に探索し、すべての `.tif` / `.tiff`（＝各撮影フォルダの各Zスライス）を読み込む
2. (このノートブック) Zスライス1枚ずつをCellposeで自動セグメンテーションする
3. (このノートブック) セグメンテーション結果を、**元のtifと同じフォルダ・同じファイル名の `.zip`**（ImageJ RoiSet形式）として保存する
4. (Fiji/ImageJ) 使うZを選び、その `画像名.zip` を対応する画像と一緒に開き、ROI Managerでおかしい細胞のROIを削除し、背景ROIを最後に追加してzipを保存し直す
5. (Fiji/ImageJ) 従来の `intensity_analysis.ijm` マクロを実行して膜/細胞質の輝度解析を行う

`intensity_analysis.ijm` は `画像名.tif` と同じフォルダにある `画像名.zip` を読みに行き、**ROI Managerの最後の1個を背景ROIとして扱う**仕様になっているため、このノートブックが出力するのは背景ROIを含まない「細胞のROIのみ」です。背景ROIの追加はこれまで通りFiji側で行ってください。

## 事前準備

- 上部メニューの **「ランタイム」→「ランタイムのタイプを変更」→ハードウェアアクセラレータで GPU (T4など) を選択**してください。
- 撮影ごとのフォルダ（各フォルダの中にZスライスのtifが入っている）を、Google Drive上の1つの親フォルダにまとめておいてください。サブフォルダ構成のままで構いません（このノートブックが再帰的に探索します）。


In [ ]:
# GPUが割り当てられているか確認
!nvidia-smi


In [ ]:
# 必要なパッケージのインストール
# - cellpose: 細胞セグメンテーション
# - roifile: ImageJ ROI (.roi / RoiSet.zip) の読み書き
!pip install -q cellpose roifile tifffile


In [ ]:
# Google Driveをマウント
from google.colab import drive
drive.mount('/content/drive')


## パス設定・画像の探索

`INPUT_DIR` を、撮影フォルダ（Zスライスtifが入ったフォルダ）をまとめてある親フォルダに変更してください。例えば

```
INPUT_DIR/
  260703-WT-PMA-60min-1408-488/
    260703-WT-PMA-60min-1408-488_C001Z001.tif
    ...
    260703-WT-PMA-60min-1408-488_C001Z007.tif
  260703-他の撮影/
    ...
```

のような構成を想定し、`INPUT_DIR` 以下を再帰的に探索してすべての `.tif`/`.tiff` を1枚ずつ処理します（Z投影はせず、Zスライスごとに個別にROIを作ります）。

`OUTPUT_DIR` は ROI (`.zip`) の保存先です。Fijiのマクロは「画像と同じフォルダにある同名の `.zip`」を探すので、迷ったら `INPUT_DIR` と同じにしておくのが安全です（デフォルトでそうなっています。各tifと同じサブフォルダに保存されます）。

もし1つの撮影フォルダに複数チャンネル（例: `C001`, `C002`）のtifが混在していて、セグメンテーションに使いたいチャンネルが1つだけの場合は、`FILENAME_FILTER` にそのチャンネルを表す文字列（例: `"C001"`）を指定すると、そのチャンネルのファイルだけを処理できます。


In [ ]:
import os

# 撮影ごとのフォルダ（Zスライスtifが入っている）をまとめた親フォルダ
INPUT_DIR = "/content/drive/MyDrive/cellpose_input"

# ROI (.zip) の保存先。Fijiのマクロが「各tifと同じフォルダの同名zip」を探すため、
# 基本はINPUT_DIRと同じにしておき、各tifと同じサブフォルダに保存する
OUTPUT_DIR = INPUT_DIR

# セグメンテーション結果を目視確認するための重ね合わせ画像 (QC画像) の保存先
# (INPUT_DIR以下のフォルダ構成をそのままミラーして保存する)
QC_DIR = os.path.join(OUTPUT_DIR, "qc_overlays")

# 複数チャンネルがあり、特定のチャンネルだけをセグメントしたい場合はファイル名に
# 含まれる文字列を指定する（例: "C001"）。すべて処理する場合は None のままでよい。
FILENAME_FILTER = None

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(QC_DIR, exist_ok=True)

# INPUT_DIR以下を再帰的に探索し、INPUT_DIRからの相対パスを集める
image_files = []
for root, dirs, files in os.walk(INPUT_DIR):
    # 前回の出力（qc_overlaysフォルダ）は探索対象から除外する
    dirs[:] = [d for d in dirs if d != os.path.basename(QC_DIR)]
    for f in files:
        if not f.lower().endswith((".tif", ".tiff")):
            continue
        if FILENAME_FILTER is not None and FILENAME_FILTER not in f:
            continue
        rel_path = os.path.relpath(os.path.join(root, f), INPUT_DIR)
        image_files.append(rel_path)

image_files = sorted(image_files)

print(f"{len(image_files)} 枚のZスライス画像が見つかりました（1枚ずつ個別にROIを作成します）")
for f in image_files:
    print(" -", f)


## Cellposeのパラメータ設定

研究室のPCでCellposeを使っていたときと同じ設定（モデルの種類・チャンネル・直径など）に合わせてください。分からない場合はデフォルト値のまま次のプレビューセルで試し、結果を見ながら調整できます。

- `MODEL_TYPE`: `"cyto3"`, `"cyto2"`, `"nuclei"` など
- `CHANNELS`: `[0, 0]` = グレースケール1チャンネルのみでセグメント。核染色などの補助チャンネルを使う場合は `[1, 2]` のように変更（Cellposeの仕様に準拠）
- `DIAMETER`: 細胞の直径(px)が分かっていれば数値を指定。`None` なら自動推定


In [ ]:
from cellpose import models
import torch

use_gpu = torch.cuda.is_available()
print("GPU使用:", use_gpu)

MODEL_TYPE = "cyto3"        # 研究室PCで使っていたモデル名に合わせて変更
DIAMETER = None              # Noneなら自動推定。既知の細胞直径(px)があれば数値を指定
CHANNELS = [0, 0]            # [0,0] = グレースケール1chで全体をセグメント
FLOW_THRESHOLD = 0.4
CELLPROB_THRESHOLD = 0.0

model = models.Cellpose(gpu=use_gpu, model_type=MODEL_TYPE)


## パラメータのプレビュー（1枚だけ試す）

一括処理の前に、まず1枚（1つのZスライス）でセグメンテーション結果を確認します。細胞が大きすぎる/小さすぎる、うまく分割できていない等があれば、上のセルで `DIAMETER` や `FLOW_THRESHOLD` を調整してこのセルを再実行してください。


In [ ]:
import tifffile
import matplotlib.pyplot as plt
import numpy as np

preview_file = image_files[0]
img = tifffile.imread(os.path.join(INPUT_DIR, preview_file))
print(f"対象ファイル: {preview_file}")
print(f"画像shape: {img.shape}, dtype: {img.dtype}")

masks, flows, styles, diams = model.eval(
    img,
    diameter=DIAMETER,
    channels=CHANNELS,
    flow_threshold=FLOW_THRESHOLD,
    cellprob_threshold=CELLPROB_THRESHOLD,
)

print(f"{preview_file}: {masks.max()} 個の細胞を検出（推定直径 {diams:.1f}px）")

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(img, cmap="gray")
ax[0].set_title("元画像")
ax[1].imshow(img, cmap="gray")
ax[1].imshow(np.ma.masked_where(masks == 0, masks), cmap="jet", alpha=0.5)
ax[1].set_title("Cellposeセグメンテーション結果")
for a in ax:
    a.axis("off")
plt.show()


## マスク → ImageJ ROI (.zip) 変換

CellposeのマスクからROI輪郭を取り出し、ImageJの `RoiSet.zip` 形式で保存する関数です。


In [ ]:
from cellpose import utils as cp_utils
import roifile

def masks_to_imagej_roi_zip(masks, save_path):
    """Cellposeのマスクを画像1枚分のImageJ RoiSet (.zip) として保存する"""
    outlines = cp_utils.outlines_list(masks)
    rois = []
    for i, outline in enumerate(outlines):
        if outline.shape[0] < 3:
            continue
        roi = roifile.ImagejRoi.frompoints(outline)
        roi.name = f"{i + 1:04d}"
        rois.append(roi)

    if os.path.exists(save_path):
        os.remove(save_path)
    roifile.roiwrite(save_path, rois)
    return len(rois)


## 一括処理

`image_files` に含まれる全Zスライスに対して個別にセグメンテーションを行い、以下を保存します（Z投影はせず、1つのtifにつき1つのzipを作成します）。

- 元のtifと同じサブフォルダの `画像名.zip`: Fijiの `intensity_analysis.ijm` がそのまま読み込めるROI（細胞のみ、背景ROIは含みません）
- `QC_DIR` 以下（元と同じサブフォルダ構成）の `画像名_qc.png`: セグメンテーション結果を目視確認するための重ね合わせ画像

処理後、QC画像やFiji上でどのZが一番良いか確認し、使うZの `.zip` だけをFijiで開いて、おかしい細胞のROIを削除、背景ROIを最後に追加してから `.zip` を保存し直してください。


In [ ]:
results_summary = []

for rel_path in image_files:
    img_path = os.path.join(INPUT_DIR, rel_path)
    img = tifffile.imread(img_path)

    masks, flows, styles, diams = model.eval(
        img,
        diameter=DIAMETER,
        channels=CHANNELS,
        flow_threshold=FLOW_THRESHOLD,
        cellprob_threshold=CELLPROB_THRESHOLD,
    )

    rel_dir = os.path.dirname(rel_path)
    basename = os.path.splitext(os.path.basename(rel_path))[0]

    # ROIは元のtifと同じサブフォルダに保存する（Fijiマクロの想定に合わせる）
    out_dir = os.path.join(OUTPUT_DIR, rel_dir)
    os.makedirs(out_dir, exist_ok=True)
    roi_path = os.path.join(out_dir, f"{basename}.zip")
    n_rois = masks_to_imagej_roi_zip(masks, roi_path)

    # 目視確認用の重ね合わせ画像を保存（フォルダ構成をミラーする）
    qc_out_dir = os.path.join(QC_DIR, rel_dir)
    os.makedirs(qc_out_dir, exist_ok=True)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(img, cmap="gray")
    ax.imshow(np.ma.masked_where(masks == 0, masks), cmap="jet", alpha=0.5)
    ax.set_title(f"{rel_path} ({n_rois} cells)")
    ax.axis("off")
    fig.savefig(os.path.join(qc_out_dir, f"{basename}_qc.png"), dpi=100, bbox_inches="tight")
    plt.close(fig)

    results_summary.append((rel_path, n_rois))
    print(f"{rel_path}: {n_rois} 個のROIを保存 -> {roi_path}")

print("\n=== 完了 ===")
for rel_path, n in results_summary:
    print(f"{rel_path}: {n} cells")


## 次のステップ (Fiji/ImageJ側の作業)

1. Google Drive for desktopなどでPCと同期するか、Driveから直接ダウンロードして、各撮影フォルダに `画像名.tif` と `画像名.zip` が同じフォルダにあることを確認する
2. `QC_DIR`（`qc_overlays`フォルダ、元と同じサブフォルダ構成）内の `_qc.png` を撮影ごとに見比べて、一番ピントが合っている・セグメンテーションが綺麗なZを選ぶ（崩れている場合はパラメータを調整して再実行）
3. Fijiで選んだZの画像を開き、ROI Managerで対応する `画像名.zip` を読み込む
4. 明らかにおかしい細胞のROIを選択して削除する
5. 背景となる領域のROIを新規作成し、**ROI Managerの一番最後に追加**する（`intensity_analysis.ijm` は最後のROIを背景として扱う仕様）
6. ROI Managerの内容を `画像名.zip` として上書き保存する
7. 従来通り `intensity_analysis.ijm` マクロを実行する（使わない他のZのtif/zipは無視してよい）
